# [v2] Trajectory classification — GELSTM (whole-brain)

**Prediction task: trajectory classification.** The model sees the *entire*
visit history of each subject and outputs one subject-level converter
probability. This is **NOT early detection** — for that, see
`EARLY_DETECTION_GELSTM_FIRST_N.ipynb`, which restricts the model to the
first N=2 or N=3 visits with `require_full_window=True`.

Sanity checks (split overlap, Δt-only baseline, shuffled-order, fixed-N,
duplicate-matrix audit) live in the four `SANITY_*` notebooks. The cell
below runs the cheapest of them (split overlap) at the head of training.


# GELSTM DELCODE Whole-Brain — Longitudinal MCI Conversion Classifier

**Architecture**: Shared GAAE encoder (per-visit) → [z_t ‖ Δt_t] → LSTM → P(converter)

Follows the same 5-fold stratified subject-level CV structure as `COST_WEIGHTED_GEC_DELCODE_WHOLE_BRAIN_REFACTORED.ipynb`.

In [ ]:
import sys
from pathlib import Path
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))


In [ ]:
import json, os, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix, classification_report
from model.GELSTM.models  import GELSTMClassifier
from model.GELSTM.dataset import LongitudinalSubjectDataset
from model.GELSTM.train   import train_epoch, evaluate, make_batches
from model.GELSTM.utils   import set_seed, compute_class_weights, compute_class_cost_weights
warnings.filterwarnings('ignore')
print('Imports OK')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Configuration

In [ ]:
RANDOM_STATE = 42
set_seed(RANDOM_STATE)

# ── Paths ────────────────────────────────────────────────────────────────
WB_DATA_ROOT  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v3__/matrices'
METADATA_DIR  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v3__/metadata'
COHORTS_CSV   = os.path.join(METADATA_DIR, 'cohorts.csv')
SPLITS_DIR    = os.path.join(METADATA_DIR, 'splits_gaae')
TRAIN_CSV     = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV       = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV      = os.path.join(SPLITS_DIR, 'test.csv')

CHECKPOINT_SEARCH_DIRS = [
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_dmn'),
]
OUTPUT_DIR = str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gelstm_whole_brain')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GAAE config (must match checkpoint) ────────────────────────────────
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'

# ── LSTM hyper-params ──────────────────────────────────────────────────
LSTM_HIDDEN      = 128
LSTM_LAYERS      = 2
LSTM_DROPOUT     = 0.3
CLASSIFIER_HIDDEN = 64
USE_TIME_DELTA   = True      # concat Δt to z_t
GRAPH_POOL       = 'mean'    # 'mean' | 'max' | 'sum'

# ── Training ──────────────────────────────────────────────────────────
FREEZE_ENCODER   = True      # True = two-stage; False = joint fine-tune
LEARNING_RATE    = 1e-3
EPOCHS           = 50
EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE       = 16
USE_CLASS_COST_WEIGHTS  = True
GRAD_CLIP        = 1.0

# ── CV ────────────────────────────────────────────────────────────────
N_FOLDS = 5

# Load existing checkpoint instead of training?
USE_EXISTING_CHECKPOINT = False

print('Config set.')


In [ ]:
# v2 split-hygiene audit — hard-fails if any subject crosses splits.
import sys
from pathlib import Path
_V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(_V2_ROOT) not in sys.path:
    sys.path.insert(0, str(_V2_ROOT))
from common.sanity import run_full_audit
if 'METADATA_DIR' in globals():
    _splits_dir = Path(METADATA_DIR) / 'splits_gaae'
    _ = run_full_audit({
        'train': str(_splits_dir / 'train.csv'),
        'val':   str(_splits_dir / 'val.csv'),
        'test':  str(_splits_dir / 'test.csv'),
    })
else:
    print('[SANITY] METADATA_DIR not defined in this notebook — skipping split audit')


## Select GAAE Checkpoint

In [ ]:
checkpoint_candidates = sorted(
    [(run_dir.name, str(run_dir / f'model_{run_dir.name}.pth'), str(run_dir))
     for ckpt_dir in CHECKPOINT_SEARCH_DIRS
     for base_path in [Path(ckpt_dir)] if base_path.is_dir()
     for run_dir in sorted(base_path.iterdir()) if run_dir.is_dir()
     if (run_dir / f'model_{run_dir.name}.pth').exists()],
    key=lambda x: x[0],
)
if not checkpoint_candidates:
    raise FileNotFoundError('No GAAE checkpoints found.')
print('Available GAAE checkpoints:')
for i,(name,_,_) in enumerate(checkpoint_candidates): print(f'  {i}: {name}')
selected_idx = int(input('Select checkpoint index: '))
GAAE_RUN_NAME, GAAE_CKPT_PATH, GAAE_RUN_DIR = checkpoint_candidates[selected_idx]
print(f'Selected: {GAAE_RUN_NAME}')


## Load GAAE Config & Build Model

In [ ]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f: hp = json.load(f)
    print('Config loaded from', CONFIG_PATH)
else:
    hp = dict(latent_dim=64, num_heads=2, cond_dim=2, dropout=0.3, adjacency_k=8, file_variant='z_transformed')
    print('Config not found – using defaults:', hp)

IN_FEATURES    = 200   # Schaefer-200 ROIs
GAAE_HIDDEN    = IN_FEATURES
GAAE_LATENT    = hp.get('latent_dim', 64)
GAAE_HEADS     = hp.get('num_heads', 2)
GAAE_COND_DIM  = hp.get('cond_dim', 2)
GAAE_DROPOUT   = hp.get('dropout', 0.3)
ADJ_K          = hp.get('adjacency_k', 8)
FILE_VARIANT   = hp.get('file_variant', 'z_transformed')

def build_model():
    m = GELSTMClassifier(
        in_features=IN_FEATURES, gaae_hidden=GAAE_HIDDEN,
        gaae_latent=GAAE_LATENT, gaae_heads=GAAE_HEADS,
        gaae_cond_dim=GAAE_COND_DIM, gaae_dropout=GAAE_DROPOUT,
        lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS,
        lstm_dropout=LSTM_DROPOUT, use_time_delta=USE_TIME_DELTA,
        classifier_hidden=CLASSIFIER_HIDDEN,
    ).to(device)
    m.load_gaae_weights(GAAE_CKPT_PATH, device=device)
    if not FREEZE_ENCODER:
        m.unfreeze_encoder()
    trainable = sum(p.numel() for p in m.get_trainable_params())
    total     = sum(p.numel() for p in m.parameters())
    print(f'Model built: trainable={trainable:,}  total={total:,}  freeze_encoder={FREEZE_ENCODER}')
    return m

_ = build_model()   # smoke-test


## Load Datasets

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# CV pool = train + val; test held out
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)
mci_pool   = cv_pool_df[cv_pool_df['diagnosis'].isin(['mci','converter'])].copy()
test_mci   = test_df[test_df['diagnosis'].isin(['mci','converter'])].copy()

print('CV pool:', mci_pool['diagnosis'].value_counts().to_dict())
print('Test:   ', test_mci['diagnosis'].value_counts().to_dict())

cv_dataset   = LongitudinalSubjectDataset(WB_DATA_ROOT, mci_pool, COHORTS_CSV,
                                          adjacency_k=ADJ_K, file_variant=FILE_VARIANT)
test_dataset = LongitudinalSubjectDataset(WB_DATA_ROOT, test_mci, COHORTS_CSV,
                                          adjacency_k=ADJ_K, file_variant=FILE_VARIANT)

cv_labels    = cv_dataset.get_labels()
cv_subject_ids = cv_dataset.get_subject_ids()
print(f'\nCV items loaded: {len(cv_dataset)}')
print(f'Test items:      {len(test_dataset)}')


## 5-Fold Stratified Subject-Level Cross-Validation

Each fold: encode sequences → train LSTM → evaluate on held-out subjects.

In [ ]:
cv_results = {'fold':[],'val_auc':[],'val_sensitivity':[],'val_specificity':[],'val_f1':[],'best_threshold':[]}
cv_histories = {'train_loss':[],'val_loss':[]}
oof_preds, oof_targets, oof_subject_ids = [], [], []

best_val_auc, best_fold, best_model_state = 0.0, -1, None
best_threshold_overall = 0.5
best_f1_threshold      = 0.5

# Pre-load all items once to avoid re-reading files per fold
print('Pre-loading all CV items...')
cv_items = [cv_dataset[i] for i in range(len(cv_dataset))]
print(f'  {len(cv_items)} items loaded')

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)

for fold, (tr_idx, va_idx) in enumerate(sgkf.split(
    cv_items, cv_labels, groups=cv_subject_ids
)):
    print('=' * 55)
    print(f'Fold {fold+1}/{N_FOLDS}  '
          f'train={len(tr_idx)} subjects  val={len(va_idx)} subjects')

    tr_items = [cv_items[i] for i in tr_idx]
    va_items = [cv_items[i] for i in va_idx]
    tr_labels = [cv_labels[i] for i in tr_idx]

    # Class weights from training fold
    if USE_CLASS_COST_WEIGHTS:
        pos_w = compute_class_weights(tr_labels, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    else:
        criterion = nn.BCEWithLogitsLoss()

    model = build_model()
    optimizer = torch.optim.Adam(model.get_trainable_params(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)

    best_fold_auc, epochs_no_improve = 0.0, 0
    best_fold_state = None
    fold_train_losses, fold_val_losses = [], []

    for epoch in range(EPOCHS):
        tr_batches = make_batches(tr_items, BATCH_SIZE, shuffle=True)
        va_batches = make_batches(va_items, BATCH_SIZE, shuffle=False)

        tr_loss = train_epoch(model, tr_batches, optimizer, criterion,
                              device, USE_TIME_DELTA, GRAPH_POOL, GRAD_CLIP)
        va_metrics = evaluate(model, va_batches, device, USE_TIME_DELTA, GRAPH_POOL)
        va_auc = va_metrics['auc']

        scheduler.step(va_auc)
        fold_train_losses.append(tr_loss)
        fold_val_losses.append(va_auc)

        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:3d}/{EPOCHS}  '
                  f'loss={tr_loss:.4f}  val_auc={va_auc:.4f}')

        if va_auc > best_fold_auc:
            best_fold_auc   = va_auc
            best_fold_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # Reload best weights and evaluate
    model.load_state_dict(best_fold_state)
    va_batches = make_batches(va_items, BATCH_SIZE, shuffle=False)
    final_va   = evaluate(model, va_batches, device, USE_TIME_DELTA, GRAPH_POOL)

    oof_preds.extend(final_va['probs'].tolist())
    oof_targets.extend(final_va['targets'].tolist())
    oof_subject_ids.extend([cv_items[i]['subject_id'] for i in va_idx])

    for k in ['auc','sensitivity','specificity','f1']:
        cv_results[f'val_{k}'].append(final_va[k])
    cv_results['best_threshold'].append(final_va['best_threshold'])
    cv_results['fold'].append(fold+1)
    cv_histories['train_loss'].append(fold_train_losses)
    cv_histories['val_loss'].append(fold_val_losses)

    print(f'  Best AUC={final_va["auc"]:.4f}  '
          f'sens={final_va["sensitivity"]:.3f}  '
          f'spec={final_va["specificity"]:.3f}  '
          f'F1={final_va["f1"]:.3f}')

    if final_va['auc'] > best_val_auc:
        best_val_auc     = final_va['auc']
        best_fold        = fold + 1
        best_model_state = best_fold_state
        best_threshold_overall = final_va['best_threshold']

# OOF F1-optimal threshold
oof_arr = np.array(oof_preds)
oof_tgt = np.array(oof_targets)
_, _, thrs = roc_curve(oof_tgt, oof_arr)
f1s = [f1_score(oof_tgt, (oof_arr>=t).astype(int), zero_division=0) for t in thrs]
best_f1_threshold = float(thrs[np.argmax(f1s)])
print(f'\nBest fold: {best_fold}  AUC={best_val_auc:.4f}')
print(f'Youden thr: {best_threshold_overall:.4f}  F1-OOF thr: {best_f1_threshold:.4f}')


## Cross-Validation Results Summary

In [ ]:
print('\nCross-Validation Summary:')
print('=' * 60)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print('-' * 60)
for metric in ['val_auc','val_sensitivity','val_specificity','val_f1']:
    v = cv_results[metric]
    print(f'{metric:<20} {np.mean(v):>10.4f} {np.std(v):>10.4f} {np.min(v):>10.4f} {np.max(v):>10.4f}')
print(f'\nBest fold: {best_fold}  AUC={best_val_auc:.4f}')
print(f'Youden thr: {best_threshold_overall:.4f}   F1-OOF thr: {best_f1_threshold:.4f}')


In [ ]:
def _oof_metrics(thr):
    p = (oof_arr >= thr).astype(int)
    if len(np.unique(oof_tgt)) > 1:
        tn,fp,fn,tp = confusion_matrix(oof_tgt, p).ravel()
    else:
        tn=fp=fn=tp=0
    sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    f1   = f1_score(oof_tgt, p, zero_division=0)
    return sens, spec, f1

y_s,y_sp,y_f1 = _oof_metrics(best_threshold_overall)
f_s,f_sp,f_f1 = _oof_metrics(best_f1_threshold)
print('OOF threshold options:')
print(f'  [1] Youden   thr={best_threshold_overall:.4f}  sens={y_s:.3f}  spec={y_sp:.3f}  F1={y_f1:.3f}')
print(f'  [2] Best-F1  thr={best_f1_threshold:.4f}  sens={f_s:.3f}  spec={f_sp:.3f}  F1={f_f1:.3f}')
choice = input('Select threshold [1=Youden (default), 2=Best-F1]: ').strip()
if choice == '2':
    ACTIVE_THRESHOLD = best_f1_threshold; THRESHOLD_METHOD = 'oof_f1'
else:
    ACTIVE_THRESHOLD = best_threshold_overall; THRESHOLD_METHOD = 'oof_youden'
print(f'Using {THRESHOLD_METHOD} threshold: {ACTIVE_THRESHOLD:.4f}')


## Save Best Model

In [ ]:
# Create the run directory and save the best model. This cell defines `run_dir`
# which downstream cells (test-eval, fold-deploy) rely on.
run_name = f"gelstm_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
run_dir  = Path(OUTPUT_DIR) / run_name
run_dir.mkdir(parents=True, exist_ok=True)

model_ckpt = run_dir / f"model_{run_name}.pth"
torch.save(best_model_state, model_ckpt)

run_summary = {
    'run_name':         run_name,
    'gaae_checkpoint':  GAAE_CKPT_PATH,
    'gaae_run_name':    GAAE_RUN_NAME,
    'cv_results':       {k: [float(x) for x in v] for k, v in cv_results.items() if k != 'fold'},
    'cv_folds':         cv_results['fold'],
    'best_fold':        int(best_fold),
    'best_val_auc':     float(best_val_auc),
    'active_threshold': float(ACTIVE_THRESHOLD),
    'threshold_method': THRESHOLD_METHOD,
    'hyperparams': {
        'lstm_hidden': LSTM_HIDDEN, 'lstm_layers': LSTM_LAYERS, 'lstm_dropout': LSTM_DROPOUT,
        'classifier_hidden': CLASSIFIER_HIDDEN, 'use_time_delta': USE_TIME_DELTA,
        'graph_pool': GRAPH_POOL, 'freeze_encoder': FREEZE_ENCODER,
        'learning_rate': LEARNING_RATE, 'epochs': EPOCHS,
        'batch_size': BATCH_SIZE, 'n_folds': N_FOLDS,
    },
}
with open(run_dir / 'run_summary.json', 'w') as f:
    json.dump(run_summary, f, indent=2)

print(f'Run directory: {run_dir}')
print(f'Model saved:   {model_ckpt}')
print(f'Summary:       {run_dir / "run_summary.json"}')


## Test-Set Evaluation

In [ ]:
eval_model = build_model()
eval_model.load_state_dict(best_model_state)
eval_model.eval()

test_items = [test_dataset[i] for i in range(len(test_dataset))]
te_batches = make_batches(test_items, BATCH_SIZE, shuffle=False)
te_metrics = evaluate(eval_model, te_batches, device, USE_TIME_DELTA, GRAPH_POOL, ACTIVE_THRESHOLD)

print('Test-Set Results')
print('=' * 40)
print(f'AUC:         {te_metrics["auc"]:.4f}')
print(f'Sensitivity: {te_metrics["sensitivity"]:.4f}')
print(f'Specificity: {te_metrics["specificity"]:.4f}')
print(f'F1:          {te_metrics["f1"]:.4f}')
print(f'Threshold:   {ACTIVE_THRESHOLD:.4f}  ({THRESHOLD_METHOD})')
print()
print(classification_report(te_metrics['targets'], (te_metrics['probs']>=ACTIVE_THRESHOLD).astype(int),
                            target_names=['stable_mci','converter']))


# ── Back-patch run_summary.json with test metrics ────────────
with open(run_dir / 'run_summary.json') as _f:
    _s = json.load(_f)
_s.update({
    'test_auc':          float(te_metrics['auc']),
    'test_sensitivity':  float(te_metrics['sensitivity']),
    'test_specificity':  float(te_metrics['specificity']),
    'test_f1':           float(te_metrics['f1']),
    'test_probabilities': [float(p) for p in te_metrics['probs']],
    'test_labels':        [int(t) for t in te_metrics['targets']],
})
with open(run_dir / 'run_summary.json', 'w') as _f:
    json.dump(_s, _f, indent=2)
print(f'Test metrics saved to {run_dir / "run_summary.json"}')


## ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Left: per-fold OOF
ax = axes[0]
fpr_oof, tpr_oof, _ = roc_curve(oof_tgt, oof_arr)
oof_auc = roc_auc_score(oof_tgt, oof_arr)
ax.plot(fpr_oof, tpr_oof, 'k-', lw=2, label=f'OOF AUC={oof_auc:.3f}')
for fold_i, auc_v in enumerate(cv_results['val_auc']):
    ax.plot([0,1],[0,1], alpha=0)   # placeholder for fold colours
    ax.annotate(f'F{fold_i+1}: {auc_v:.3f}', xy=(0.55, 0.15 + fold_i*0.06), fontsize=8)
ax.plot([0,1],[0,1],'--',color='gray',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('OOF ROC (all folds combined)'); ax.legend(); ax.grid(alpha=0.3)
# Right: test set
ax2 = axes[1]
te_fpr, te_tpr, _ = roc_curve(te_metrics['targets'], te_metrics['probs'])
ax2.plot(te_fpr, te_tpr, 'r-', lw=2, label=f'Test AUC={te_metrics["auc"]:.3f}')
ax2.plot([0,1],[0,1],'--',color='gray',lw=1)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('Test-Set ROC'); ax2.legend(); ax2.grid(alpha=0.3)
fig.suptitle(f'GELSTM  |  GAAE: {GAAE_RUN_NAME}', fontsize=11)
plt.tight_layout(); plt.show()


## Stratified Evaluation by Number of Scans

Evaluates model performance separately for subjects with 1, 2-3, and ≥4 scans.

In [ ]:
strat_records = []
for item, prob, tgt in zip(test_items, te_metrics['probs'], te_metrics['targets']):
    n = item['n_scans']
    if n == 1:   bucket = '1 scan'
    elif n <= 3: bucket = '2-3 scans'
    else:        bucket = '>=4 scans'
    strat_records.append({'n_scans': n, 'bucket': bucket,
                          'prob': prob, 'target': int(tgt),
                          'pred': int(prob >= ACTIVE_THRESHOLD)})

strat_df = pd.DataFrame(strat_records)
print('Stratified test-set results:')
print(f"{'Bucket':<12} {'n':>5} {'AUC':>8} {'Sens':>8} {'Spec':>8} {'F1':>8}")
for bucket in ['1 scan','2-3 scans','>=4 scans']:
    sub = strat_df[strat_df['bucket']==bucket]
    if len(sub) < 2 or sub['target'].nunique() < 2:
        print(f'{bucket:<12} {len(sub):>5}   (insufficient data)')
        continue
    auc_ = roc_auc_score(sub['target'], sub['prob'])
    tn_,fp_,fn_,tp_ = confusion_matrix(sub['target'], sub['pred']).ravel()
    sens_ = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0.0
    spec_ = tn_/(tn_+fp_) if (tn_+fp_)>0 else 0.0
    f1_   = f1_score(sub['target'], sub['pred'], zero_division=0)
    print(f'{bucket:<12} {len(sub):>5} {auc_:>8.3f} {sens_:>8.3f} {spec_:>8.3f} {f1_:>8.3f}')


## Conversion Probability Trajectories

For each test subject with ≥2 scans, plot P(converter) at each visit.

In [ ]:
# Encode each test subject visit-by-visit to get per-visit probabilities
traj_records = []
eval_model.eval()
with torch.no_grad():
    for item in test_items:
        if item['n_scans'] < 2:
            continue
        pid   = item['subject_id']
        label = item['label']
        months = item['visit_months']
        deltas = item['delta_t']
        graphs = item['graphs']
        # Encode prefix sequences of length 1, 2, ..., T
        for t in range(1, len(graphs) + 1):
            sub_item = {**item, 'graphs': graphs[:t],
                        'delta_t': deltas[:t],
                        'visit_months': months[:t], 'n_scans': t}
            packed, _, _ = __import__(
                'model.GELSTM.utils', fromlist=['encode_batch_sequences']
            ).encode_batch_sequences(
                [sub_item], eval_model, device, USE_TIME_DELTA, GRAPH_POOL)
            logit = eval_model(packed)
            prob  = torch.sigmoid(logit).item()
            traj_records.append({'pid': pid, 'label': label,
                                 'month': months[t-1], 'prob': prob})

traj_df = pd.DataFrame(traj_records)
print(f'Trajectory data: {traj_df["pid"].nunique()} subjects')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = {1: '#F44336', 0: '#2196F3'}
for ax, (label, title) in zip(axes, [(1,'Converters'),(0,'Stable MCI')]):
    sub = traj_df[traj_df['label']==label]
    for pid, grp in sub.groupby('pid'):
        ax.plot(grp['month'], grp['prob'], marker='o', alpha=0.6,
                color=palette[label], lw=1.5)
    ax.axhline(ACTIVE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold={ACTIVE_THRESHOLD:.3f}')
    ax.set_xlabel('Visit month')
    ax.set_ylabel('P(converter)')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle(f'Per-visit conversion probability trajectories  |  GELSTM {GAAE_RUN_NAME}', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Deploy the best model into the 5 fold slots the dashboard expects.
# (Degraded ensemble: all five fold files are identical copies of the best model.
# For a real per-fold ensemble, modify the CV loop to save each fold's state.)
import shutil, pathlib, json
dest = pathlib.Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/model/GELSTM/checkpoints')
dest.mkdir(parents=True, exist_ok=True)

for fold_idx in range(1, 6):
    shutil.copy(model_ckpt, dest / f'best_model_fold{fold_idx}.pth')

# The dashboard's GELSTM service searches for the GAAE encoder INSIDE
# the checkpoints/ directory (not its parent), so copy it there.
shutil.copy(GAAE_CKPT_PATH, dest / 'gaae_encoder.pth')

# Write model_card.json so the dashboard instantiates the same arch we trained.
# (Without this, gelstm.py uses defaults gaae_heads=4/dropout=0.2 which would
# mismatch our heads=2/dropout=0.3 checkpoints and fail to load.)
model_card = {
    'arch': {
        'in_features':      IN_FEATURES,
        'gaae_hidden':      GAAE_HIDDEN,
        'gaae_latent':      GAAE_LATENT,
        'gaae_heads':       GAAE_HEADS,
        'gaae_cond_dim':    GAAE_COND_DIM,
        'gaae_dropout':     GAAE_DROPOUT,
        'lstm_hidden':      LSTM_HIDDEN,
        'lstm_layers':      LSTM_LAYERS,
        'lstm_dropout':     LSTM_DROPOUT,
        'use_time_delta':   USE_TIME_DELTA,
        'classifier_hidden': CLASSIFIER_HIDDEN,
    },
    'norm': {},
}
(dest / 'model_card.json').write_text(json.dumps(model_card, indent=2))

print('Deployed:')
for p in sorted(dest.glob('*.pth')):
    print(f'  {p}')
print(f'  {dest / "model_card.json"}')
